# tp-05d ハイパーパラメータ調整（Optuna、上位モデルのみ）

**実行場所: ホスト PC（stg）。** tp-05b で上位だった LightGBM（二値）・XGBoost（二値）・小さい NN を、Optuna（TPE）で調整する。

## 設計
- **調整に使う期間**: valid 2022・2023 の 2 fold、1 seed（粗く探す）。**valid 2024 は調整に使わず、確認専用**（選択に使った期間と確認の期間を分けて、過剰適合を抑える）。
- **目的関数**: p_win・p_top2・p_top3 の LogLoss を基準モデル（既定パラメータ）の値で割って重み付き平均（0.5 / 0.25 / 0.25）。1.0 未満なら基準より良い。
- **試行数**: LightGBM 40、XGBoost 30、NN 20。study は SQLite に保存して中断から再開できる（`optuna/*.db`）。
- **調整後の評価**: 最良パラメータで fold 3 つ × seed 3 つの OOF を作り直し（`oof/model_oof/*_tuned.parquet`）、既定パラメータと比べる。2024 の改善を「確認」として別に示す。
- **記録**: MLflow（`keiba-tp-hpo`）に study 名・git コミット・特徴量リストのハッシュ・各 trial の params と fold スコア・処理時間を残す。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, hashlib, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow
import optuna
import psutil

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, zoo_runner
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "hpo"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "optuna").mkdir(exist_ok=True)
THREADS = detect_compute(reserve_cores=6).threads
SEEDS = [0, 1, 2]
mlflow.set_experiment("keiba-tp-hpo")
require_tracking_server()
optuna.logging.set_verbosity(optuna.logging.WARNING)
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
FEATS = D["FEATS"]["base+S+C1"]
X = df[FEATS]
FEAT_HASH = hashlib.md5("|".join(FEATS).encode()).hexdigest()[:10]
TUNE_FOLDS = folds[:2]            # valid 2022・2023（調整に使う）
print(f"行 {len(df):,} / 特徴量 {len(FEATS)} (hash {FEAT_HASH}) / 調整 fold {[f.fold_id for f in TUNE_FOLDS]} / threads {THREADS}")

In [ ]:
W = (0.5, 0.25, 0.25)
BASEREF = {}     # 既定パラメータでの調整 fold の LogLoss（目的関数の分母）


def score(res, ref):
    m = res[["logloss_win", "logloss_top2", "logloss_top3"]].mean()
    return float(sum(w * m[k] / ref[k] for w, k in zip(W, ("logloss_win", "logloss_top2", "logloss_top3"))))


def space_lgb(t):
    return {"lgb": {"learning_rate": t.suggest_float("learning_rate", 0.03, 0.08, log=True), "num_leaves": t.suggest_int("num_leaves", 8, 127, log=True),
                    "min_data_in_leaf": t.suggest_int("min_data_in_leaf", 20, 600, log=True), "feature_fraction": t.suggest_float("feature_fraction", 0.3, 1.0),
                    "bagging_fraction": t.suggest_float("bagging_fraction", 0.5, 1.0), "lambda_l2": t.suggest_float("lambda_l2", 0.1, 100.0, log=True)},
            "n_rounds": t.suggest_int("n_rounds", 150, 800)}


def space_xgb(t):
    return {"device": "cuda", "max_depth": t.suggest_int("max_depth", 3, 9), "learning_rate": t.suggest_float("learning_rate", 0.02, 0.1, log=True),
            "n_estimators": t.suggest_int("n_estimators", 150, 800), "subsample": t.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": t.suggest_float("colsample_bytree", 0.3, 1.0), "reg_lambda": t.suggest_float("reg_lambda", 0.1, 100.0, log=True),
            "min_child_weight": t.suggest_float("min_child_weight", 1.0, 200.0, log=True)}


def space_nn(t):
    h = t.suggest_categorical("hidden", ["64-32", "128-64", "128"])
    return {"hidden": tuple(int(x) for x in h.split("-")), "epochs": t.suggest_int("epochs", 6, 20), "lr": t.suggest_float("lr", 5e-4, 5e-3, log=True),
            "wd": t.suggest_float("wd", 1e-5, 1e-2, log=True), "dropout": t.suggest_float("dropout", 0.0, 0.4)}


TARGETS = {  # 名前: (recipe, 探索空間, 試行数, 既定パラメータ)
    "lgb": ("lgb_binary", space_lgb, 40, {}),
    "xgb": ("xgb_binary", space_xgb, 30, {"device": "cuda"}),
    "nn": ("nn", space_nn, 20, {"hidden": (128, 64), "epochs": 14}),
}
BEST = {}
for key, (recipe, space, n_trials, default) in TARGETS.items():
    t0 = time.time()
    ref_res, _ = zoo_runner.run_oof(f"{key}_ref", recipe, X, meta, TUNE_FOLDS, [0], THREADS, params=dict(default))
    ref = ref_res[["logloss_win", "logloss_top2", "logloss_top3"]].mean().to_dict()
    BASEREF[key] = ref
    study = optuna.create_study(study_name=f"tp05d_{key}", storage=f"sqlite:///{OUT / 'optuna' / (key + '.db')}", load_if_exists=True, direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=0))
    run_name = f"tp-05d-{key}"
    with mlflow.start_run(run_name=run_name, tags={"notebook": "tp-05d", "git_commit": GIT_COMMIT, "model": key, "feature_list_hash": FEAT_HASH, "study": study.study_name,
                                                    "tuning_folds": "valid_2022,valid_2023", "confirm_fold": "valid_2024"}):
        def objective(trial):
            p = space(trial)
            t1 = time.time()
            res, _ = zoo_runner.run_oof(f"{key}_trial", recipe, X, meta, TUNE_FOLDS, [0], THREADS, params=p)
            s = score(res, ref)
            for _, r in res.iterrows():
                trial.set_user_attr(f"{r['fold']}_win", float(r["logloss_win"]))
            trial.set_user_attr("seconds", time.time() - t1)
            mlflow.log_metric("trial_score", s, step=trial.number)
            return s

        remaining = max(0, n_trials - len([t for t in study.trials if t.state.name == "COMPLETE"]))
        if remaining:
            study.optimize(objective, n_trials=remaining, gc_after_trial=True)
        BEST[key] = {"params": study.best_trial.params, "score": study.best_value, "trial": study.best_trial.number}
        mlflow.log_params({f"best_{k}": str(v) for k, v in study.best_trial.params.items()})
        mlflow.log_metrics({"best_score": study.best_value, "n_trials": len(study.trials), "seconds": time.time() - t0})
        study.trials_dataframe().to_csv(OUT / f"{key}_trials.csv", index=False)
        mlflow.log_artifact(str(OUT / f"{key}_trials.csv"), "hpo")
    print(f"  {key}: best score {study.best_value:.5f}（既定 = 1.0）/ {len(study.trials)} trials / {time.time() - t0:.0f}s / best {study.best_trial.params}")
    gc.collect()
(OUT / "best_params.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "feature_hash": FEAT_HASH, "best": BEST, "ref": BASEREF}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")

In [ ]:
# 調整後のパラメータで fold 3 つ × seed 3 つの OOF を作り直し、既定と比べる（valid 2024 は調整に使っていない確認）
def to_params(key, p):
    if key == "lgb":
        return {"lgb": {k: p[k] for k in ("learning_rate", "num_leaves", "min_data_in_leaf", "feature_fraction", "bagging_fraction", "lambda_l2")}, "n_rounds": p["n_rounds"]}
    if key == "xgb":
        return {"device": "cuda", **p}
    return {"hidden": tuple(int(x) for x in p["hidden"].split("-")), **{k: p[k] for k in ("epochs", "lr", "wd", "dropout")}}


FINAL = []
for key, (recipe, _, _, default) in TARGETS.items():
    with mlflow.start_run(run_name=f"tp-05d-final-{key}", tags={"notebook": "tp-05d", "git_commit": GIT_COMMIT, "model": key, "stage": "final_eval"}):
        res_t, _ = zoo_runner.run_oof(f"{key}_tuned", recipe, X, meta, folds, SEEDS, THREADS, params=to_params(key, BEST[key]["params"]), out_dir=OOF_DIR)
        res_d, _ = zoo_runner.run_oof(f"{key}_default", recipe, X, meta, folds, SEEDS, THREADS, params=dict(default), out_dir=None)
        for r in (res_t, res_d):
            FINAL.append(r)
        mlflow.log_metrics({f"tuned_{c}": float(res_t[c].mean()) for c in ("logloss_win", "logloss_top2", "logloss_top3", "race_nll")})
        mlflow.log_metrics({f"default_{c}": float(res_d[c].mean()) for c in ("logloss_win", "logloss_top2", "logloss_top3", "race_nll")})
        mlflow.log_metrics({f"valid2024_delta_{t}": float(res_t[res_t.fold == 'valid_2024'][f'logloss_{t}'].iloc[0] - res_d[res_d.fold == 'valid_2024'][f'logloss_{t}'].iloc[0]) for t in ("win", "top2", "top3")})
    gc.collect()
F = pd.concat(FINAL, ignore_index=True)
F.to_csv(OUT / "final_fold_metrics.csv", index=False)
tab = F.groupby("model")[["logloss_win", "logloss_top2", "logloss_top3", "race_nll", "auc_win", "top1_hit"]].mean()
display(tab.round(5))
conf = F[F.fold == "valid_2024"].set_index("model")[["logloss_win", "logloss_top2", "logloss_top3"]]
print("確認 fold（valid 2024、調整に未使用）の LogLoss: 調整後 − 既定（負なら改善）")
cmp = pd.DataFrame({k: (conf.loc[f"{k}_tuned"] - conf.loc[f"{k}_default"]) for k in TARGETS}).T
display(cmp.round(5)); cmp.to_csv(OUT / "confirm_2024_delta.csv"); tab.to_csv(OUT / "final_summary.csv")